# Intron AfriSpeech-200 Automatic Speech Recognition Challenge
**Objective**: Create an automatic speech recognition (ASR) model for African accents.
**Metric**: Word Error Rate (WER)

In [ ]:
import os
import torch
import torchaudio
import pandas as pd
from datasets import Dataset, load_metric
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor, Trainer, TrainingArguments
import numpy as np
import re
import json

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Path Configuration

In [ ]:
def setup_paths():
    base_dir = '/kaggle/input'
    train_csv = 'train.csv'
    audio_dir = 'audio'
    
    if os.path.exists(base_dir):
        for root, dirs, files in os.walk(base_dir):
            if 'train.csv' in files:
                train_csv = os.path.join(root, 'train.csv')
            if 'audio' in dirs:
                audio_dir = os.path.join(root, 'audio')
                
    return train_csv, audio_dir

TRAIN_CSV, AUDIO_DIR = setup_paths()
print(f"CSV Path: {TRAIN_CSV}")
print(f"Audio Directory: {AUDIO_DIR}")

## 2. Data Preparation Utils

In [ ]:
def remove_special_characters(batch):
    batch["transcription"] = re.sub(r'[\,\.\!\?\-\;\:\"\(\)]', '', batch["transcription"]).lower() + " "
    return batch

def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_values"] = processor(audio["array"], sampling_rate=audio["sampling_rate"]).input_values[0]
    
    with processor.as_target_processor():
        batch["labels"] = processor(batch["transcription"]).input_ids
    return batch

## 3. Model and Trainer Setup

In [ ]:
MODEL_ID = "facebook/wav2vec2-large-xlsr-53"

try:
    processor = Wav2Vec2Processor.from_pretrained(MODEL_ID)
    model = Wav2Vec2ForCTC.from_pretrained(
        MODEL_ID, 
        attention_dropout=0.1,
        hidden_dropout=0.1,
        feat_proj_dropout=0.0,
        mask_time_prob=0.05,
        layerdrop=0.1,
        gradient_checkpointing=True, 
        ctc_loss_reduction="mean", 
        pad_token_id=processor.tokenizer.pad_token_id,
        vocab_size=len(processor.tokenizer)
    ).to(device)
except Exception as e:
    print(f"Error loading model: {e}")

## 4. Evaluation Function

In [ ]:
wer_metric = load_metric("wer")

def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)
    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

## 5. Main Training Execution

In [ ]:
if os.path.exists(TRAIN_CSV):
    df = pd.read_csv(TRAIN_CSV)
    # Assuming columns 'path' and 'transcription'
    # Note: On Kaggle, you might need to adjust 'path' to include AUDIO_DIR
    df['path'] = df['path'].apply(lambda x: os.path.join(AUDIO_DIR, os.path.basename(x)))
    
    # Create Dataset object from pandas
    # (Only using a small subset for demonstration purposes)
    dataset = Dataset.from_pandas(df.head(100))
    
    # Clean text
    dataset = dataset.map(remove_special_characters)
    
    # Cast audio column to audio type
    from datasets import Audio
    dataset = dataset.cast_column("path", Audio(sampling_rate=16_000))
    dataset = dataset.rename_column("path", "audio")
    
    # Preprocess
    dataset = dataset.map(prepare_dataset, remove_columns=dataset.column_names, num_proc=4)
    
    # Split
    dataset = dataset.train_test_split(test_size=0.1)
    
    training_args = TrainingArguments(
      output_dir="./wav2vec2-afrispeech",
      group_by_length=True,
      per_device_train_batch_size=8,
      gradient_accumulation_steps=2,
      evaluation_strategy="steps",
      num_train_epochs=10,
      fp16=True,
      save_steps=500,
      eval_steps=500,
      logging_steps=500,
      learning_rate=3e-4,
      warmup_steps=500,
      save_total_limit=2,
    )
    
    # Initialize Trainer
    from transformers import DataCollatorForCTCWithPadding
    data_collator = DataCollatorForCTCWithPadding(processor=processor, padding=True)
    
    trainer = Trainer(
        model=model,
        data_collator=data_collator,
        args=training_args,
        compute_metrics=compute_metrics,
        train_dataset=dataset["train"],
        eval_dataset=dataset["test"],
        tokenizer=processor.feature_extractor,
    )
    
    print("Starting training...")
    # trainer.train()  # Uncomment to start actual training
else:
    print("Train.csv not found. Please attach the 'AfriSpeech' dataset on Kaggle.")